# DDR Bandwidth Benchmark — PYNQ Notebook

This notebook demonstrates how to use the `ddr_bw_top` hardware overlay to measure
PS DDR memory bandwidth from the PL side on a KV260 (or any PYNQ-compatible Zynq board).

## What it does
1. Loads the `ddr_bw.bit` bitstream onto the PL.
2. Allocates four DDR buffers (one per AXI master port).
3. Configures the benchmark engine via AXI-Lite registers.
4. Runs read, write, and mixed read/write benchmarks.
5. Reports per-port and aggregate bandwidth from both hardware counters and wall-clock time.

## Prerequisites
- PYNQ image installed on KV260 (or equivalent)
- `ddr_bw.bit` and `ddr_bw.hwh` bitstream files in the same directory
- Python packages: `pynq`, `numpy`

In [ ]:
import sys, os
# Add the python driver directory to path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'python'))

import numpy as np
from ddr_bw_benchmark import DDRBandwidthBenchmark, BenchmarkConfig
from ddr_bw_regs import (
    MODE_READ, MODE_WRITE, MODE_RD_WR_SPLIT,
    print_register_map
)

print('Imports successful')

## 1. Register Map Reference

Print the complete register map for the `ddr_bw_top` IP block.

In [ ]:
print_register_map()

## 2. Load Bitstream

Load the `ddr_bw.bit` bitstream. **Update the path** to match your board.

In [ ]:
BITFILE = 'ddr_bw.bit'  # Update this path as needed
PL_CLK_MHZ = 300.0      # KV260 pl_clk0 default; adjust if different

bm = DDRBandwidthBenchmark(BITFILE)
bm.load()

## 3. Read Bandwidth Test

All 4 AXI master ports issue sequential read bursts from DDR.

In [ ]:
cfg_read = BenchmarkConfig(
    mode=MODE_READ,
    burst_len=16,           # 16 beats × 8 bytes = 128 bytes per burst
    total_bursts=4096,      # per port: 4096 × 128B = 512 KB per port
    buf_size_bytes=4 * 1024 * 1024,  # 4 MB buffer per port
    clk_freq_mhz=PL_CLK_MHZ,
    timeout_s=30.0,
)

results_read = bm.run(cfg_read)
DDRBandwidthBenchmark.print_results(results_read)

## 4. Write Bandwidth Test

All 4 AXI master ports issue sequential write bursts to DDR.

In [ ]:
cfg_write = BenchmarkConfig(
    mode=MODE_WRITE,
    burst_len=16,
    total_bursts=4096,
    buf_size_bytes=4 * 1024 * 1024,
    clk_freq_mhz=PL_CLK_MHZ,
    timeout_s=30.0,
)

results_write = bm.run(cfg_write)
DDRBandwidthBenchmark.print_results(results_write)

## 5. Mixed Read/Write Bandwidth Test

Ports 0 and 1 issue reads; ports 2 and 3 issue writes simultaneously.

In [ ]:
cfg_mixed = BenchmarkConfig(
    mode=MODE_RD_WR_SPLIT,
    burst_len=16,
    total_bursts=4096,
    buf_size_bytes=4 * 1024 * 1024,
    clk_freq_mhz=PL_CLK_MHZ,
    timeout_s=30.0,
)

results_mixed = bm.run(cfg_mixed)
DDRBandwidthBenchmark.print_results(results_mixed)

## 6. Sweep Burst Length

Measure how bandwidth scales with burst length for READ mode.

In [ ]:
burst_lengths = [1, 2, 4, 8, 16, 32, 64, 128, 255]
hw_bw_results = []
sw_bw_results = []

for bl in burst_lengths:
    cfg = BenchmarkConfig(
        mode=MODE_READ,
        burst_len=bl,
        total_bursts=1024,
        buf_size_bytes=max(4 * 1024 * 1024, bl * 1024 * 8 * 2),
        clk_freq_mhz=PL_CLK_MHZ,
        timeout_s=30.0,
    )
    r = bm.run(cfg)
    hw_bw_results.append(r.hw_aggregate_bw_gbps)
    sw_bw_results.append(r.sw_aggregate_bw_gbps)
    print(f'  burst_len={bl:4d}  HW={r.hw_aggregate_bw_gbps:.3f} GB/s  '
          f'SW={r.sw_aggregate_bw_gbps:.3f} GB/s')

print('Sweep complete.')

## 7. Bandwidth vs Burst Length Plot

Visualize the sweep results.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(burst_lengths, hw_bw_results, 'o-', label='HW bandwidth (cycle counter)')
ax.plot(burst_lengths, sw_bw_results, 's--', label='SW bandwidth (wall-clock)')
ax.axhline(y=19.2, color='r', linestyle=':', label='KV260 DDR4 peak (~19.2 GB/s)')
ax.set_xlabel('AXI Burst Length (beats)')
ax.set_ylabel('Aggregate Read Bandwidth (GB/s)')
ax.set_title('DDR Read Bandwidth vs Burst Length (4 ports, 64-bit data width)')
ax.legend()
ax.set_xscale('log', base=2)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('bandwidth_sweep.png', dpi=150)
plt.show()
print('Plot saved to bandwidth_sweep.png')

## 8. How Bandwidth is Calculated

### Hardware bandwidth (cycle counter)

The `ddr_bw_counter` module counts PL clock cycles from the first `start_pulse`
until all 4 ports have set their `done` flags:

```
HW_time_s = CYCLE_CNT / clk_freq_Hz
bytes_port = P_BEAT_CNT × AXI_DATA_WIDTH_BYTES  (8 bytes for 64-bit bus)
total_bytes = sum(bytes_port[0:4])
HW_BW_GB_s = total_bytes / HW_time_s / 1e9
```

### Software bandwidth (wall-clock)

Measured with `time.perf_counter()` from just before the `CTRL_START` write
to when the `STATUS_ALL_DONE` flag is polled as set:

```
SW_BW_GB_s = total_bytes / wall_time_s / 1e9
```

The software bandwidth includes Python polling overhead, which is why it is
typically a few percent lower than the hardware measurement.

### KV260 DDR4 theoretical peak

KV260 uses DDR4-2400 in single-channel configuration:
- 2400 MHz × 64-bit bus = 19.2 GB/s peak
- Practical achievable bandwidth from PL: 60–85% of peak depending on burst
  length, access pattern, and whether the PS CPUs are simultaneously accessing DDR.

In [ ]:
# Summary comparison table
print('Summary: READ / WRITE / MIXED bandwidth comparison')
print(f'{"Mode":<16} {"HW Agg (GB/s)":>16} {"SW Agg (GB/s)":>16} {"HW Time (ms)":>14}')
print('-' * 65)
for name, r in [("READ", results_read), ("WRITE", results_write), ("RD_WR_SPLIT", results_mixed)]:
    print(f'{name:<16} {r.hw_aggregate_bw_gbps:>16.3f} {r.sw_aggregate_bw_gbps:>16.3f} {r.hw_time_s*1000:>14.3f}')

## 9. Cleanup

In [ ]:
bm.close()